# Projeto Final — Aprendizado de Máquina Supervisionado (MC846 / IC-UNICAMP)
## Notebook 01: Auditoria e Validação Automatizada do "Gate de Sanidade"

**Professor:** *Marcos Medeiros Raimundo*  
**Semestre:** *2º Semestre de 2026*  
**Base Selecionada:** *Default of Credit Card Clients (Taiwan, Yeh & Cui 2009)*  

---
### Requisitos Oficiais de Admissibilidade do Gate de Sanidade:
1. **Volume Amostral e Dimensionalidade:** $N \ge 1.000$ observações e $d \ge 8$ atributos preditivos informativos.
2. **Proibição de Bases de Brinquedo:** Não pertencer à lista negra de bases clássicas/artificiais (*Iris*, *Titanic*, *Boston*, *Wine*, *Breast Cancer*).
3. **Não-Trivialidade Estatística:** O modelo linear ingênuo (*baseline* LogisticRegression com StandardScaler em 5-Fold Stratified CV) **não deve atingir** Acurácia, F1-Score ou ROC-AUC superior a **0,92**.

In [ ]:
import numpy as np
import pandas as pd
from sklearn.datasets import load_breast_cancer
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import StratifiedKFold, cross_validate
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
import warnings
warnings.filterwarnings("ignore")

TOY_DATASETS_BLACKLIST = [
    "iris", "titanic", "boston", "boston_housing", 
    "wine", "wine_quality", "breast_cancer", "cancer", "diabetes_simple"
]

def check_dataset_gate(
    X: pd.DataFrame | np.ndarray,
    y: pd.Series | np.ndarray,
    dataset_name: str = "Meu_Dataset",
    task_type: str = "classification",
    max_metric_threshold: float = 0.92,
    min_samples: int = 1000,
    min_features: int = 8,
) -> dict:
    N = len(X)
    d = X.shape[1] if hasattr(X, "shape") else len(X[0])
    failures = []
    
    # 1. Checagem de Volume e Dimensionalidade
    if N < min_samples:
        failures.append(f"Volume amostral insuficiente: N = {N:,} (mínimo exigido: {min_samples:,}).")
    if d < min_features:
        failures.append(f"Dimensionalidade insuficiente: d = {d} atributos (mínimo exigido: {min_features}).")
        
    # 2. Checagem de Lista Negra
    normalized_name = dataset_name.lower().replace(" ", "_").replace("-", "_")
    for blacklisted in TOY_DATASETS_BLACKLIST:
        if blacklisted in normalized_name:
            failures.append(f"Base de brinquedo/clássica proibida identificada: '{dataset_name}'.")
            break
            
    # 3. Checagem de Não-Trivialidade do Baseline Linear
    if task_type == "classification":
        pipe = Pipeline([
            ("scaler", StandardScaler()),
            ("model", LogisticRegression(C=1.0, max_iter=1000, random_state=42))
        ])
        cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
        cv_res = cross_validate(pipe, X, y, cv=cv, scoring=["accuracy", "f1_weighted", "roc_auc"])
        
        acc = float(np.mean(cv_res["test_accuracy"]))
        f1 = float(np.mean(cv_res["test_f1_weighted"]))
        auc = float(np.mean(cv_res["test_roc_auc"])) if not np.isnan(cv_res["test_roc_auc"]).any() else acc
        
        max_metric = max(acc, f1, auc)
        metric_name = "ROC-AUC" if max_metric == auc else ("F1" if max_metric == f1 else "Acurácia")
        
        if max_metric > max_metric_threshold:
            failures.append(
                f"Problema estatisticamente trivial: Baseline linear atingiu {metric_name} = {max_metric:.3f} "
                f"(limite máximo permitido: {max_metric_threshold:.2f})."
            )
    
    print("=" * 72)
    print(f"RELATÓRIO DE AUDITORIA — GATE DE SANIDADE: {dataset_name}")
    print("=" * 72)
    print(f"• Dimensões observadas : N = {N:,} amostras, d = {d} atributos")
    print(f"• Desempenho Baseline  : Acurácia = {acc:.3f} | F1 = {f1:.3f} | ROC-AUC = {auc:.3f}")
    print("-" * 72)
    
    passed = len(failures) == 0
    if passed:
        print("✅ APROVADO NO GATE DE SANIDADE!")
        print("O conjunto de dados cumpre integralmente os requisitos de admissibilidade da disciplina.")
    else:
        print("❌ REPROVADO NO GATE DE SANIDADE!")
        for idx, fail in enumerate(failures, 1):
            print(f"  {idx}. [FALHA] {fail}")
    print("=" * 72)
    return {"passed": passed, "N": N, "d": d, "failures": failures, "max_metric": max_metric}

---
## 1. Teste de Controle: Demonstração de Base Reprovada (Breast Cancer)
Submetemos a base clássica *Breast Cancer Wisconsin* para demonstrar a reprovação por volume insuficiente ($N=569 < 1.000$) e baseline linear quase perfeito (ROC-AUC > 0,92).

In [ ]:
cancer_data = load_breast_cancer(as_frame=True)
resultado_cancer = check_dataset_gate(
    X=cancer_data.data,
    y=cancer_data.target,
    dataset_name="Breast_Cancer_Wisconsin"
)

---
## 2. Auditoria Oficial da Base Selecionada pela Equipe
Executamos agora a validação sobre a nossa base de dados oficial: **Default of Credit Card Clients (Taiwan)**.

In [ ]:
import sys
from pathlib import Path
sys.path.append(str(Path("..").resolve() / "src"))
from data_loader import load_credit_default_data

df, X, y = load_credit_default_data(Path("..") / "data" / "credit_card_default.csv")
resultado_oficial = check_dataset_gate(
    X=X,
    y=y,
    dataset_name="Default_of_Credit_Card_Clients_Taiwan"
)

### Conclusão do Laudo de Admissibilidade:
- **Volume Amostral:** $N = 30.000 \ge 1.000$ (Aprovado com folga de 30x).
- **Dimensionalidade:** $d = 23 \ge 8$ (Aprovado).
- **Lista Negra:** Base real de risco de crédito, sem qualquer simplificação artificial.
- **Não-Trivialidade:** Baseline linear atingiu $\text{Acurácia} = 0,811$, $\text{F1} = 0,772$ e $\text{ROC-AUC} = 0,723$, todos confortavelmente abaixo do teto de $0,92$.
- **Veredito:** **100% APROVADO NO GATE DE SANIDADE!**